In [3]:
import pandas as pd
import sqlite3

# load your CSVs
df_t = pd.read_csv("../data/agg_transactions.csv")
df_u = pd.read_csv("../data/agg_users.csv")

# clean state names
df_t['state'] = df_t['state'].str.replace('-', ' ').str.title()
df_u['state'] = df_u['state'].str.replace('-', ' ').str.title()

# add amount in crores
df_t['amount_cr'] = df_t['amount'] / 1e7

# create SQLite database — this creates a file called phonepe.db
conn = sqlite3.connect("../data/phonepe.db")

# write tables into the database
df_t.to_sql("transactions", conn, if_exists="replace", index=False)
df_u.to_sql("users", conn, if_exists="replace", index=False)

print("Database created successfully")
print("Tables:", pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))

Database created successfully
Tables:            name
0  transactions
1         users


In [4]:
def query(sql):
    return pd.read_sql(sql, conn)

# Q1: Which states generate the most transaction value?

In [5]:
query("""
    SELECT 
        state,
        ROUND(SUM(amount_cr), 2) AS total_amount_cr,
        SUM(count) AS total_transactions,
        ROUND(SUM(amount_cr) * 100.0 / (SELECT SUM(amount_cr) FROM transactions), 2) AS pct_of_india
    FROM transactions
    GROUP BY state
    ORDER BY total_amount_cr DESC
    LIMIT 10
""")

,state,total_amount_cr,total_transactions,pct_of_india
0,Telangana,4165595.56,26174684592,12.06
1,Karnataka,4067872.18,30970946279,11.77
2,Maharashtra,4037419.57,31985208732,11.68
3,Andhra Pradesh,3466908.05,18918696723,10.03
4,Uttar Pradesh,2688521.22,18523603727,7.78
5,Rajasthan,2634323.56,17108537898,7.62
6,Madhya Pradesh,1912527.94,14072176059,5.54
7,Bihar,1790134.93,10941026824,5.18
8,West Bengal,1558416.43,9191499687,4.51
9,Odisha,1226398.21,8918527452,3.55


# Q2: What is the year-over-year growth in transaction volume?

In [6]:
query("""
    SELECT
        year,
        SUM(count) AS total_transactions,
        ROUND(SUM(amount_cr), 0) AS total_amount_cr,
        ROUND(
            (SUM(amount_cr) - LAG(SUM(amount_cr)) OVER (ORDER BY year)) 
            * 100.0 / LAG(SUM(amount_cr)) OVER (ORDER BY year), 
        2) AS yoy_growth_pct
    FROM transactions
    GROUP BY year
    ORDER BY year
""")

,year,total_transactions,total_amount_cr,yoy_growth_pct
0,2018,1080202410,162305.0,NaN
1,2019,4079827215,627669.0,286.72
2,2020,7973974741,1464116.0,133.26
3,2021,19288429220,3459870.0,136.31
4,2022,39301293734,6426633.0,85.75
5,2023,64257054687,9449181.0,47.03
6,2024,99303434867,12962455.0,37.18


# Q3: Which transaction type dominates in each state?

In [7]:
query("""
    WITH ranked AS (
        SELECT
            state,
            transaction_type,
            SUM(count) AS total_count,
            RANK() OVER (PARTITION BY state ORDER BY SUM(count) DESC) AS rnk
        FROM transactions
        GROUP BY state, transaction_type
    )
    SELECT state, transaction_type, total_count
    FROM ranked
    WHERE rnk = 1
    ORDER BY total_count DESC
""")

,state,transaction_type,total_count
0,Maharashtra,Merchant payments,19290166803
1,Karnataka,Merchant payments,18851053952
2,Telangana,Merchant payments,14953817184
3,Uttar Pradesh,Merchant payments,9959659787
4,Rajasthan,Merchant payments,9262580721
5,Andhra Pradesh,Merchant payments,9234938613
6,Madhya Pradesh,Merchant payments,7820161481
7,Bihar,Merchant payments,5268451146
8,Delhi,Merchant payments,5060364831
9,Odisha,Merchant payments,4569191284


# Q4: Which states are growing fastest? (YoY by state)

In [8]:
query("""
    WITH yearly AS (
        SELECT
            state,
            year,
            SUM(amount_cr) AS amount_cr
        FROM transactions
        GROUP BY state, year
    ),
    with_growth AS (
        SELECT
            state,
            year,
            amount_cr,
            LAG(amount_cr) OVER (PARTITION BY state ORDER BY year) AS prev_amount
        FROM yearly
    )
    SELECT
        state,
        year,
        ROUND(amount_cr, 1) AS amount_cr,
        ROUND((amount_cr - prev_amount) * 100.0 / prev_amount, 1) AS yoy_growth_pct
    FROM with_growth
    WHERE year = (SELECT MAX(year) FROM transactions)
        AND prev_amount IS NOT NULL
    ORDER BY yoy_growth_pct DESC
    LIMIT 10
""")

,state,year,amount_cr,yoy_growth_pct
0,Manipur,2024,4387.4,87.0
1,Lakshadweep,2024,71.6,80.2
2,Jammu & Kashmir,2024,52844.4,69.6
3,Bihar,2024,740379.1,52.7
4,Ladakh,2024,3971.8,51.4
5,West Bengal,2024,638711.3,51.1
6,Arunachal Pradesh,2024,12243.5,50.3
7,Meghalaya,2024,6839.2,47.2
8,Andaman & Nicobar Islands,2024,3074.0,46.8
9,Chhattisgarh,2024,203351.2,46.4


# Q5: Festive quarter analysis (Q3 vs other quarters)

In [9]:
query("""
    SELECT
        quarter,
        ROUND(AVG(quarterly_amount), 0) AS avg_amount_cr,
        ROUND(SUM(quarterly_amount), 0) AS total_amount_cr,
        ROUND(
            AVG(quarterly_amount) * 100.0 / 
            (SELECT AVG(sub.quarterly_amount) 
             FROM (SELECT SUM(amount_cr) AS quarterly_amount 
                   FROM transactions GROUP BY year, quarter) sub),
        1) AS pct_vs_avg
    FROM (
        SELECT year, quarter, SUM(amount_cr) AS quarterly_amount
        FROM transactions
        GROUP BY year, quarter
    )
    GROUP BY quarter
    ORDER BY quarter
""")

,quarter,avg_amount_cr,total_amount_cr,pct_vs_avg
0,1,1047932.0,7335523.0,84.9
1,2,1171775.0,8202423.0,95.0
2,3,1260473.0,8823314.0,102.1
3,4,1455853.0,10190969.0,118.0


# Q6: States with high users but low transactions (churn risk)

In [10]:
query("""
    WITH state_totals AS (
        SELECT
            t.state,
            SUM(t.count) AS total_transactions,
            SUM(u.registered_users) AS total_users,
            ROUND(CAST(SUM(t.count) AS FLOAT) / SUM(u.registered_users), 2) AS txn_per_user
        FROM transactions t
        JOIN users u ON t.state = u.state AND t.year = u.year AND t.quarter = u.quarter
        GROUP BY t.state
    )
    SELECT
        state,
        total_transactions,
        total_users,
        txn_per_user,
        CASE
            WHEN txn_per_user < 5 THEN 'High churn risk'
            WHEN txn_per_user BETWEEN 5 AND 15 THEN 'Average engagement'
            ELSE 'Highly engaged'
        END AS engagement_label
    FROM state_totals
    ORDER BY txn_per_user ASC
""")

,state,total_transactions,total_users,txn_per_user,engagement_label
0,Lakshadweep,883994,704014,1.26,High churn risk
1,Mizoram,19535750,13505305,1.45,High churn risk
2,Manipur,73350340,46945655,1.56,High churn risk
3,Nagaland,54768743,31488375,1.74,High churn risk
4,Tripura,120838496,69313135,1.74,High churn risk
5,Himachal Pradesh,500810554,248999125,2.01,High churn risk
6,Punjab,1772412497,857161255,2.07,High churn risk
7,Kerala,2300897800,1023790375,2.25,High churn risk
8,Sikkim,65860842,24340020,2.71,High churn risk
9,Meghalaya,88635419,30926410,2.87,High churn risk


# Q7: Top 5 states per year (ranking changes over time)

In [11]:
query("""
    WITH yearly_state AS (
        SELECT
            state,
            year,
            SUM(amount_cr) AS amount_cr,
            RANK() OVER (PARTITION BY year ORDER BY SUM(amount_cr) DESC) AS rnk
        FROM transactions
        GROUP BY state, year
    )
    SELECT state, year, ROUND(amount_cr, 0) AS amount_cr, rnk
    FROM yearly_state
    WHERE rnk <= 5
    ORDER BY year, rnk
""")

,state,year,amount_cr,rnk
0,Maharashtra,2018,18985.0,1
1,Karnataka,2018,17397.0,2
2,Uttar Pradesh,2018,13177.0,3
3,Andhra Pradesh,2018,12207.0,4
4,Telangana,2018,11786.0,5
5,Karnataka,2019,79126.0,1
6,Maharashtra,2019,73733.0,2
7,Telangana,2019,73388.0,3
8,Andhra Pradesh,2019,53779.0,4
9,Uttar Pradesh,2019,43225.0,5


In [12]:
# closing the connection
conn.close()
print("All queries done. Database saved at ../data/phonepe.db")

All queries done. Database saved at ../data/phonepe.db
